In [ ]:
import sys
sys.path.insert(0, '../iaml')

import pandas as pd

from iaml import *
from IPython.display import display, Markdown as IMarkdown, HTML as IHTML
from rich.console import Console
from rich.markdown import Markdown

In [2]:
dataset = pd.read_csv('../perf_logger/tests_data/titanic.csv', delimiter=';')

In [ ]:
dataset.shape

In [ ]:
autom = IAML(max_duration=120)

print(autom.default_pipeline())
print(autom.json_pipeline())

In [ ]:
pipeline = {
    'step': 'MetaOrderedStep',
    'children': [
        {
            'step': 'MetaStep',
            'tag': 'cleaning',
        },
        {
            'step': 'MetaStep',
            'tag': 'features_selection',
        },
        # {
        #    'step': 'WrapKFold',
        #    'children': [{
            #    'step': 'ActKNN',
        #    }]
        # },
    ]
}

# autom.load_pipeline(pipeline)
print(autom.json_pipeline())


In [ ]:
results = autom.fit(
    dataset.drop('label', axis=1).copy(),
    dataset[['label']]).copy()

In [ ]:
test = dataset.iloc[200:(200+68)].drop('label', axis=1).copy()
md = autom.chosen_candidate.pipeline.explain_model(test)
ret = md.to_binary_plots()

In [ ]:
ret = md.to_binary_plots()

In [ ]:
test = dataset.iloc[200:(200+68)].drop('label', axis=1).copy()
md = autom.chosen_candidate.pipeline.explain_model(test)
display(IMarkdown(md.to_markdown_plots()))

In [ ]:
print(results[0].pipeline.model)
print(results[0].pipeline.steps)

console = Console()

for step in results[0].pipeline.explanations:
    if step:
        display(IMarkdown(step))

test = dataset.iloc[200:(200+68)].drop('label', axis=1).copy()

pm = results[0].pipeline.pickle()

md = autom.chosen_candidate.pipeline.explain_model(test)
display(IMarkdown(md.to_markdown(plots=['force', 'waterfall', 'scatter', 'beeswarm', 'heatmap', 'bar'])))

In [ ]:
display(IMarkdown(md.to_markdown_plots()))

In [ ]:
md.to_b64_plots()

In [ ]:
import pickle

o = 0 # offset
n = 68  # # of samples
labels = dataset.iloc[o:(o+n)]['label']
predict_df = dataset.iloc[o:(o+n)].drop('label', axis=1).copy()

# labels = labels.reset_index()
predict_df.reset_index(inplace=True, drop=True)

m = pickle.loads(pm)
sum([ r == labels.iloc[o+i] for i, r in enumerate(m.predict(predict_df)) ]) / n

In [ ]:
final_boss_iaml = IAML(max_workers=2)
final_boss_iaml.default_pipeline()
final_boss_results = final_boss_iaml.fit(dataset.drop('label', axis=1).copy(), dataset[['label']].copy())

In [ ]:
# sum([ len(r.model.pickle()) for r in final_boss_results ])
[ (r.model.ml_model, r.evaluate()) for r in final_boss_results if r.model.ml_model is not None ]